
# JogjaPulse - Dataset Profile & Quality Check

**Purpose:** profile every file in `RawDataset/` and flag anything that needs cleaning or that affects integration - missing rows, wrong formats, out-of-range values, duplicates, and cross-file consistency.

**Important framing:** this is a *published, privacy-protected aggregate*. Many "missing rows" are **by design** (k=5 suppression; see README S5): a missing row means "fewer than 5 devices", **not zero, not data loss**. This notebook separates structural gaps (suppression + zero-device slots) from genuine format problems.

**How to use:** run top to bottom. Adjust `DATA_DIR` in the setup cell if your path differs. A `RESULTS` dict collects a verdict printed at the end.


In [ ]:

import os, re
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# Resolve data dir robustly (notebook lives in IndividualCode/1_ExploreCleaning/)
CANDIDATES = [
    r"d:\UGM Things\Semester 5\Data Engineering\Projects\DE1Assignment_JogjaPulse\RawDataset",
    os.path.abspath("../../RawDataset"),
    os.path.abspath("../RawDataset"),
    os.path.abspath("RawDataset"),
]
DATA_DIR = next((p for p in CANDIDATES if os.path.isdir(p)), CANDIDATES[0])
print("DATA_DIR =", DATA_DIR)

EXPECTED_DATES = pd.date_range("2021-12-01", "2022-04-30", freq="D")   # 151 days
EXPECTED_MONTHS = ["2021-12", "2022-01", "2022-02", "2022-03", "2022-04"]
EXPECTED_KAB = ["KULON PROGO", "BANTUL", "GUNUNGKIDUL", "SLEMAN", "YOGYAKARTA"]
N_KAPANEWON = 78

RESULTS = {}

def path(name):
    return os.path.join(DATA_DIR, name)

def profile_df(df, name):
    print(f"\n--- {name} ---")
    print("shape:", df.shape)
    print("dtypes:\n", df.dtypes.to_string())
    miss = df.isna().sum()
    miss = miss[miss > 0]
    print("missing NaN per col:", "none" if miss.empty else miss.to_dict())
    obj = df.select_dtypes(include=["object", "string"])
    empties = {c: int((df[c].astype(str).str.strip() == "").sum()) for c in obj.columns}
    empties = {c: n for c, n in empties.items() if n}
    print("empty-string cells:", "none" if not empties else empties)
    print("fully duplicate rows:", int(df.duplicated().sum()))



## 1. `presence_cell500m_daily.csv` - distinct devices per 500m grid cell per day

Watch for: date completeness (all 151 days), `cell_id` format `E{i}_N{j}` and its consistency with `x_min_utm`/`y_min_utm` (divisible by 500), `n_maids >= 5`, lon/lat inside DIY, and the size of the suppressed/zero-device gap vs the full grid. This file intentionally has **no `n_pings`** column.


In [ ]:

df = pd.read_csv(path("presence_cell500m_daily.csv"))
profile_df(df, "presence_cell500m_daily.csv")

print("columns:", list(df.columns))
bad = df[~df["cell_id"].str.match(r"^E\d+_N\d+$")]
print("bad cell_id rows:", len(bad))
exp = "E" + (df["x_min_utm"] // 500).astype(int).astype(str) + "_N" + (df["y_min_utm"] // 500).astype(int).astype(str)
print("cell_id mismatch vs x_min/500,y_min/500:", int((exp != df["cell_id"]).sum()))
print("x not divisible by 500:", int((df["x_min_utm"] % 500 != 0).sum()), "| y not divisible by 500:", int((df["y_min_utm"] % 500 != 0).sum()))
d = pd.to_datetime(df["date"])
print("date range:", d.min().date(), "->", d.max().date(), "| distinct days:", d.nunique())
missing_days = sorted(set(EXPECTED_DATES.date) - set(d.dt.date))
print("missing calendar days:", missing_days if missing_days else "none")
print("distinct cells:", df["cell_id"].nunique())
print("n_maids: min", df["n_maids"].min(), "max", df["n_maids"].max(), "| <5:", int((df["n_maids"] < 5).sum()), "| <=0:", int((df["n_maids"] <= 0).sum()))
print("lon range:", df.centroid_lon.min(), df.centroid_lon.max(), "| lat range:", df.centroid_lat.min(), df.centroid_lat.max())
full = df["cell_id"].nunique() * d.nunique()
print(f"full grid {full} vs published {len(df)} -> missing {full - len(df)} ({(full - len(df)) / full * 100:.1f}%) [suppression + zero-device slots]")

RESULTS["presence_cell500m_daily"] = {
    "rows": int(len(df)),
    "distinct_days": int(d.nunique()),
    "missing_days": missing_days if missing_days else "none",
    "distinct_cells": int(df["cell_id"].nunique()),
    "bad_cell_id_rows": int(len(bad)),
    "cell_id_xy_mismatch": int((exp != df["cell_id"]).sum()),
    "n_maids_min": int(df["n_maids"].min()),
    "n_maids_below_5": int((df["n_maids"] < 5).sum()),
    "duplicate_rows": int(df.duplicated().sum()),
    "grid_gap_pct": round((full - len(df)) / full * 100, 1),
}



## 2. `presence_kapanewon_hourly.csv` - distinct devices + pings per kapanewon per hour

Watch for: 78 kapanewon, 5 kab_kota with expected names, date completeness, hours 0-23 all present, `n_maids >= 5`, and `n_pings >= n_maids` (presence requires >=1 ping). The gap vs the full 78 x 151 x 24 grid is suppression + zero-device slots.


In [ ]:

df = pd.read_csv(path("presence_kapanewon_hourly.csv"), dtype={"kab_code": str})
profile_df(df, "presence_kapanewon_hourly.csv")

print("columns:", list(df.columns))
print("kab_kota set:", sorted(df.kab_kota.unique()))
print("unexpected kab_kota:", set(df.kab_kota) - set(EXPECTED_KAB))
print("distinct kapanewon:", df.kapanewon.nunique(), "| distinct kapanewon_code:", df.kapanewon_code.nunique())
name_collisions = df.groupby("kapanewon").kapanewon_code.nunique()
name_collisions = name_collisions[name_collisions > 1]
print("kapanewon names shared by >1 code:", name_collisions.to_dict() if len(name_collisions) else "none")
print("kab_code format bad:", int((~df.kab_code.str.match(r"^\d{2}\.\d{2}$")).sum()))
print("kapanewon_code format bad:", int((~df.kapanewon_code.str.match(r"^\d{2}\.\d{2}\.\d{2}$")).sum()))
d = pd.to_datetime(df["date"])
print("date range:", d.min().date(), "->", d.max().date(), "| distinct days:", d.nunique())
missing_days = sorted(set(EXPECTED_DATES.date) - set(d.dt.date))
print("missing calendar days:", missing_days if missing_days else "none")
print("hour set:", sorted(df.hour.unique()), "| out of 0-23 rows:", len(df[~df.hour.between(0, 23)]))
print("n_maids: min", df.n_maids.min(), "max", df.n_maids.max(), "| <5:", int((df.n_maids < 5).sum()))
print("n_pings: min", df.n_pings.min(), "| n_pings<n_maids:", int((df.n_pings < df.n_maids).sum()), "| n_pings<=0:", int((df.n_pings <= 0).sum()))
print("dup on (kapanewon_code,date,hour):", int(df.duplicated(["kapanewon_code", "date", "hour"]).sum()))
full = df.kapanewon.nunique() * d.nunique() * 24
print(f"full grid {full} vs published {len(df)} -> missing {full - len(df)} ({(full - len(df)) / full * 100:.1f}%) [suppression + zero-device slots]")

RESULTS["presence_kapanewon_hourly"] = {
    "rows": int(len(df)),
    "distinct_days": int(d.nunique()),
    "missing_days": missing_days if missing_days else "none",
    "distinct_kapanewon_names": int(df.kapanewon.nunique()),
    "distinct_kapanewon_codes": int(df.kapanewon_code.nunique()),
    "name_collisions": name_collisions.to_dict() if len(name_collisions) else "none",
    "kab_kota_ok": set(df.kab_kota) == set(EXPECTED_KAB),
    "hours_ok": sorted(df.hour.unique()) == list(range(24)),
    "n_maids_min": int(df.n_maids.min()),
    "n_pings_lt_n_maids": int((df.n_pings < df.n_maids).sum()),
    "duplicate_keys": int(df.duplicated(["kapanewon_code", "date", "hour"]).sum()),
    "grid_gap_pct": round((full - len(df)) / full * 100, 1),
}



## 3. `home_maids_vs_population_kapanewon.csv` - devices by home kapanewon vs official population

Watch for: exactly 78 rows, unique kapanewon, no NaN/inf in `ratio`, `pop_actual > 0`, and `ratio == n_users / pop_actual`.


In [ ]:

df = pd.read_csv(path("home_maids_vs_population_kapanewon.csv"))
profile_df(df, "home_maids_vs_population_kapanewon.csv")

print("rows == 78:", len(df) == 78, "(", len(df), ")")
print("kapanewon unique:", df.KEC_KAPN.nunique(), "| dup:", int(df.KEC_KAPN.duplicated().sum()))
print("ratio NaN:", int(df.ratio.isna().sum()), "| inf:", int(np.isinf(df.ratio).sum()))
print("pop_actual <=0:", int((df.pop_actual <= 0).sum()))
chk = np.isclose(df.ratio, df.n_users / df.pop_actual)
print("ratio mismatch vs n_users/pop_actual:", int((~chk).sum()))
print("ratio range:", df.ratio.min(), df.ratio.max())

RESULTS["home_maids_vs_population_kapanewon"] = {
    "rows": int(len(df)),
    "rows_eq_78": len(df) == 78,
    "kapanewon_unique": int(df.KEC_KAPN.nunique()),
    "kapanewon_dup": int(df.KEC_KAPN.duplicated().sum()),
    "ratio_nan": int(df.ratio.isna().sum()),
    "ratio_inf": int(np.isinf(df.ratio).sum()),
    "ratio_mismatch": int((~chk).sum()),
    "pop_le0": int((df.pop_actual <= 0).sum()),
}



## 4. `validation_night_presence_kapanewon.csv` - night-time presence (validation)

Watch for: 156 rows = 78 kapanewon x 2 periods, both expected periods present, `slots_published_pct` in 0-100.


In [ ]:

df = pd.read_csv(path("validation_night_presence_kapanewon.csv"))
profile_df(df, "validation_night_presence_kapanewon.csv")

print("rows:", len(df), "| periods:", sorted(df.period.unique()))
print("kapanewon count per period:\n", df.groupby("period").kapanewon.nunique())
print("night_presence NaN:", int(df.night_presence.isna().sum()), "| slots_published_pct range:", df.slots_published_pct.min(), df.slots_published_pct.max())
print("kapanewon total unique:", df.kapanewon.nunique())

RESULTS["validation_night_presence_kapanewon"] = {
    "rows": int(len(df)),
    "rows_eq_156": len(df) == 156,
    "periods": sorted(df.period.unique()),
    "kapanewon_total": int(df.kapanewon.nunique()),
    "night_presence_nan": int(df.night_presence.isna().sum()),
    "slots_pct_range": [float(df.slots_published_pct.min()), float(df.slots_published_pct.max())],
}



## 5. `tier1_summary.csv` - suppression statistics for the two presence products

Watch for: 2 rows, products `A_cell500m_daily` and `B_kapanewon_hourly`, and that `rows_published` matches the *actual* row counts of files 1 and 2 (cross-file check).


In [ ]:

df = pd.read_csv(path("tier1_summary.csv"))
profile_df(df, "tier1_summary.csv")
print(df.to_string())
print("products:", list(df.product))
cell = pd.read_csv(path("presence_cell500m_daily.csv"), usecols=[0])
kap = pd.read_csv(path("presence_kapanewon_hourly.csv"), usecols=[0])
a = int(df.loc[df.product == "A_cell500m_daily", "rows_published"].iloc[0])
b = int(df.loc[df.product == "B_kapanewon_hourly", "rows_published"].iloc[0])
print(f"A rows_published {a} vs actual {len(cell)} -> {'OK' if a == len(cell) else 'MISMATCH'}")
print(f"B rows_published {b} vs actual {len(kap)} -> {'OK' if b == len(kap) else 'MISMATCH'}")

RESULTS["tier1_summary"] = {
    "rows": int(len(df)),
    "products": list(df.product),
    "A_rows_published_vs_actual": "OK" if a == len(cell) else f"MISMATCH ({a} vs {len(cell)})",
    "B_rows_published_vs_actual": "OK" if b == len(kap) else f"MISMATCH ({b} vs {len(kap)})",
}



## 6. `od_home_visits_kapanewon_monthly.csv` - home-based visitation flows (monthly)

Watch for: 5 months, `reduced_quality=True` only for 2022-03 and 2022-04, code format `dd.dd.dd`, `n_maids >= 5`, unique (month, origin, dest) keys, and that intra-zonal rows (origin == dest) are present.


In [ ]:

df = pd.read_csv(path("od_home_visits_kapanewon_monthly.csv"), dtype={"origin_kab_code": str, "dest_kab_code": str})
profile_df(df, "od_home_visits_kapanewon_monthly.csv")

print("months:", sorted(df.month.unique()))
print("missing months:", set(EXPECTED_MONTHS) - set(df.month.unique()) or "none")
print("month format bad:", int((~df.month.str.match(r"^\d{4}-\d{2}$")).sum()))
print("origin_code format bad:", int((~df.origin_code.str.match(r"^\d{2}\.\d{2}\.\d{2}$")).sum()))
print("dest_code format bad:", int((~df.dest_code.str.match(r"^\d{2}\.\d{2}\.\d{2}$")).sum()))
print("distinct origin:", df.origin_code.nunique(), "| distinct dest:", df.dest_code.nunique())
print("reduced_quality True months:", sorted(df.loc[df.reduced_quality, "month"].unique()))
print("reduced_quality False months:", sorted(df.loc[~df.reduced_quality, "month"].unique()))
print("n_maids: min", df.n_maids.min(), "| <5:", int((df.n_maids < 5).sum()))
print("dup on (month,origin_code,dest_code):", int(df.duplicated(["month", "origin_code", "dest_code"]).sum()))
print("intra-zonal (origin==dest) rows:", int((df.origin_code == df.dest_code).sum()))
print("reduced_quality dtype:", df.reduced_quality.dtype, "| values:", df.reduced_quality.unique())

RESULTS["od_home_visits_kapanewon_monthly"] = {
    "rows": int(len(df)),
    "months": sorted(df.month.unique()),
    "missing_months": list(set(EXPECTED_MONTHS) - set(df.month.unique())),
    "distinct_origin": int(df.origin_code.nunique()),
    "distinct_dest": int(df.dest_code.nunique()),
    "reduced_quality_months": sorted(df.loc[df.reduced_quality, "month"].unique()),
    "n_maids_min": int(df.n_maids.min()),
    "duplicate_keys": int(df.duplicated(["month", "origin_code", "dest_code"]).sum()),
    "intra_zonal_rows": int((df.origin_code == df.dest_code).sum()),
}



## 7. `od_origin_homes_monthly.csv` - devices homed per origin kapanewon per month (OD denominators)

Watch for: 390 rows = 78 kapanewon x 5 months, `reduced_quality` flag consistent with file 6, and `n_homed_maids_daytime` NaN where the count is below 5 (by design).


In [ ]:

df = pd.read_csv(path("od_origin_homes_monthly.csv"), dtype={"origin_kab_code": str})
profile_df(df, "od_origin_homes_monthly.csv")

print("months:", sorted(df.month.unique()))
print("rows:", len(df), "| expected 78*5=390 ->", len(df) == 390)
print("origin kapanewon per month:\n", df.groupby("month").origin_code.nunique())
print("n_homed_maids <5:", int((df.n_homed_maids < 5).sum()))
print("n_homed_maids_daytime NaN (expected when <5):", int(df.n_homed_maids_daytime.isna().sum()))
print("reduced_quality True months:", sorted(df.loc[df.reduced_quality, "month"].unique()))

RESULTS["od_origin_homes_monthly"] = {
    "rows": int(len(df)),
    "rows_eq_390": len(df) == 390,
    "months": sorted(df.month.unique()),
    "origin_per_month": df.groupby("month").origin_code.nunique().to_dict(),
    "n_homed_maids_below_5": int((df.n_homed_maids < 5).sum()),
    "daytime_nan": int(df.n_homed_maids_daytime.isna().sum()),
    "reduced_quality_months": sorted(df.loc[df.reduced_quality, "month"].unique()),
}



## 8. `od_summary.csv` - suppression statistics for the OD product

Watch for: 1 row, `k=5`, `min_nights=2`, and that `od_rows_published` / `origin_rows_published` match the actual row counts of files 6 and 7.


In [ ]:

df = pd.read_csv(path("od_summary.csv"))
profile_df(df, "od_summary.csv")
print(df.to_string())
odv = pd.read_csv(path("od_home_visits_kapanewon_monthly.csv"), usecols=[0])
odh = pd.read_csv(path("od_origin_homes_monthly.csv"), usecols=[0])
op = int(df.od_rows_published.iloc[0])
orp = int(df.origin_rows_published.iloc[0])
print(f"od_rows_published {op} vs actual od_home_visits {len(odv)} -> {'OK' if op == len(odv) else 'MISMATCH'}")
print(f"origin_rows_published {orp} vs actual od_origin_homes {len(odh)} -> {'OK' if orp == len(odh) else 'MISMATCH'}")

RESULTS["od_summary"] = {
    "rows": int(len(df)),
    "k": int(df.k.iloc[0]),
    "min_nights": int(df.min_nights.iloc[0]),
    "od_rows_published_vs_actual": "OK" if op == len(odv) else f"MISMATCH ({op} vs {len(odv)})",
    "origin_rows_published_vs_actual": "OK" if orp == len(odh) else f"MISMATCH ({orp} vs {len(odh)})",
}



## 9. Cross-file consistency

The 78 kapanewon should be the *same set* across every file that uses kapanewon names. If they differ, integration (joins/merges) will silently drop rows.


In [ ]:

pk = pd.read_csv(path("presence_kapanewon_hourly.csv"), usecols=["kapanewon"])
hm = pd.read_csv(path("home_maids_vs_population_kapanewon.csv"), usecols=["KEC_KAPN"])
vn = pd.read_csv(path("validation_night_presence_kapanewon.csv"), usecols=["kapanewon"])
odv = pd.read_csv(path("od_home_visits_kapanewon_monthly.csv"), usecols=["origin_kapanewon", "dest_kapanewon"])
odh = pd.read_csv(path("od_origin_homes_monthly.csv"), usecols=["origin_kapanewon"])
set_pk = set(pk.kapanewon); set_hm = set(hm.KEC_KAPN); set_vn = set(vn.kapanewon)
set_od = set(odv.origin_kapanewon) | set(odv.dest_kapanewon) | set(odh.origin_kapanewon)
print("kapanewon counts: presence", len(set_pk), "| home", len(set_hm), "| validation", len(set_vn), "| od", len(set_od))
print("presence==home:", set_pk == set_hm, "| presence==validation:", set_pk == set_vn, "| presence==od:", set_pk == set_od)
print("only in presence:", set_pk - set_hm, "| only in home:", set_hm - set_pk)
RESULTS["_kapanewon_consistency"] = {
    "presence": len(set_pk), "home": len(set_hm), "validation": len(set_vn), "od": len(set_od),
    "all_match": set_pk == set_hm == set_vn == set_od,
}



## 10. Verdict summary

Compiled from the `RESULTS` dict collected above. Read this first for a quick "is it clean?" answer; the per-file cells above have the details.


In [ ]:

print("=" * 72)
print("VERDICT SUMMARY")
print("=" * 72)
for name, info in RESULTS.items():
    print(f"\n## {name}")
    for k, v in info.items():
        print(f"   {k}: {v}")
